In [1]:
!cp -r /kaggle/input/comotion/ml-comotion-main /kaggle/working/


In [2]:
# ============================================
# CELL 1: SETUP COMOTION (run once)
# ============================================

# Copy CoMotion code
!cp -r /kaggle/input/comotion/ml-comotion-main /kaggle/working/

# Copy SMPL model
!cp /kaggle/input/smpl-model/SMPL_python_v.1.1.0/smpl/models/basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl \
    /kaggle/working/ml-comotion-main/src/comotion_demo/data/smpl/SMPL_NEUTRAL.pkl

# Install dependencies
!pip install smplx -q

# Install CoMotion
%cd /kaggle/working/ml-comotion-main
!pip install -e . -q

# Download pretrained models
!bash get_pretrained_models.sh

print("✅ CoMotion setup complete!")

/kaggle/working/ml-comotion-main
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 998.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.0 MB/s eta 0:00:00
   ━

In [3]:
# ============================================
# CELL 2: RUN COMOTION ON VIDEO
# ============================================
VIDEO_NAME = "sample_swing2"  
!cp /kaggle/input/project-with-model/FYP/Swing-motion-Analysis/Data/{VIDEO_NAME}.mp4 /kaggle/working/

VIDEO_PATH = f"/kaggle/working/{VIDEO_NAME}.mp4"
OUTPUT_DIR = "/kaggle/working/results"

# Run CoMotion
!python /kaggle/working/ml-comotion-main/demo.py \
    -i {VIDEO_PATH} \
    -o {OUTPUT_DIR} \
    --skip-visualization

print(f"✅ CoMotion processing complete for {VIDEO_NAME}")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
# ============================================
# CELL 3: EXTRACT 3D JOINTS FROM SMPL
# ============================================

import torch
import numpy as np
from smplx import SMPL

VIDEO_NAME = "sample_swing2" 
PT_FILE = f"/kaggle/working/results/{VIDEO_NAME}.pt"

# Load CoMotion output
data = torch.load(PT_FILE, map_location='cpu')
print(f"✅ Loaded: {PT_FILE}")
print(f"   Keys: {list(data.keys())}")
print(f"   Frames: {data['pose'].shape[0]}")

# Extract tensors
pose_tensor = data['pose']
trans_tensor = data['trans']
betas_tensor = data['betas']
global_orient = pose_tensor[:, :3]
body_pose = pose_tensor[:, 3:72]

# Initialize SMPL model
smpl_model = SMPL(
    model_path='/kaggle/working/ml-comotion-main/src/comotion_demo/data/smpl',
    gender='neutral',
    create_transl=True
).to('cpu')
print("✅ SMPL model initialized")

# Extract joints for all frames
print("\n⏳ Extracting joints...")
all_joints = []

for frame_idx in range(pose_tensor.shape[0]):
    if frame_idx % 20 == 0:
        print(f"  Frame {frame_idx}/{pose_tensor.shape[0]}")
    
    smpl_out = smpl_model(
        body_pose=body_pose[frame_idx].unsqueeze(0),
        global_orient=global_orient[frame_idx].unsqueeze(0),
        betas=betas_tensor[frame_idx].unsqueeze(0),
        transl=trans_tensor[frame_idx].unsqueeze(0)
    )
    
    joints = smpl_out.joints[0].detach().cpu().numpy()
    all_joints.append(joints)

all_joints = np.array(all_joints)
print(f"\n✅ Joints extracted: {all_joints.shape}")  # (num_frames, 45, 3)

# Save for later use
np.save(f"/kaggle/working/results/{VIDEO_NAME}_joints.npy", all_joints)
print(f"✅ Saved: /kaggle/working/results/{VIDEO_NAME}_joints.npy")

/tmp/ipykernel_24/533631429.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(PT_FILE, map_location='cpu')


✅ Loaded: /kaggle/working/results/sample_swing2.pt
   Keys: ['id', 'pose', 'trans', 'betas', 'frame_idx']
   Frames: 323
✅ SMPL model initialized

⏳ Extracting joints...
  Frame 0/323
  Frame 20/323
  Frame 40/323
  Frame 60/323
  Frame 80/323
  Frame 100/323
  Frame 120/323
  Frame 140/323
  Frame 160/323
  Frame 180/323
  Frame 200/323
  Frame 220/323
  Frame 240/323
  Frame 260/323
  Frame 280/323
  Frame 300/323
  Frame 320/323

✅ Joints extracted: (323, 45, 3)
✅ Saved: /kaggle/working/results/sample_swing2_joints.npy


In [5]:
# ============================================
# CELL 4: CONVERT TO SWING ANALYSIS FORMAT
# ============================================

import numpy as np
import pandas as pd

VIDEO_NAME = "sample_swing2"

# Load joints
all_joints = np.load(f"/kaggle/working/results/{VIDEO_NAME}_joints.npy")
print(f"✅ Loaded joints: {all_joints.shape}")

# SMPL Joint Indices (24 body joints + 21 hand joints = 45 total)
# We use the main body joints:
JOINT_MAP = {
    'pelvis': 0,
    'left_hip': 1,
    'right_hip': 2,
    'spine1': 3,
    'left_knee': 4,
    'right_knee': 5,
    'spine2': 6,
    'left_ankle': 7,
    'right_ankle': 8,
    'spine3': 9,
    'left_foot': 10,
    'right_foot': 11,
    'neck': 12,
    'left_collar': 13,
    'right_collar': 14,
    'head': 15,
    'left_shoulder': 16,
    'right_shoulder': 17,
    'left_elbow': 18,
    'right_elbow': 19,
    'left_wrist': 20,
    'right_wrist': 21,
    'left_hand': 22,
    'right_hand': 23,
}

# Create DataFrame matching MediaPipe format
pose_data = []

for frame_idx in range(all_joints.shape[0]):
    joints = all_joints[frame_idx]
    
    # Use LEFT side (lead arm for right-handed golfer)
    # Swap to RIGHT side if analyzing left-handed golfer
    frame_data = {
        "frame": frame_idx,
        
        # Wrist
        "wrist_x": joints[JOINT_MAP['left_wrist'], 0],
        "wrist_y": joints[JOINT_MAP['left_wrist'], 1],
        "wrist_z": joints[JOINT_MAP['left_wrist'], 2],
        "wrist_visibility": 1.0,
        
        # Elbow
        "elbow_x": joints[JOINT_MAP['left_elbow'], 0],
        "elbow_y": joints[JOINT_MAP['left_elbow'], 1],
        "elbow_z": joints[JOINT_MAP['left_elbow'], 2],
        "elbow_visibility": 1.0,
        
        # Shoulder
        "shoulder_x": joints[JOINT_MAP['left_shoulder'], 0],
        "shoulder_y": joints[JOINT_MAP['left_shoulder'], 1],
        "shoulder_z": joints[JOINT_MAP['left_shoulder'], 2],
        "shoulder_visibility": 1.0,
        
        # Hip
        "hip_x": joints[JOINT_MAP['left_hip'], 0],
        "hip_y": joints[JOINT_MAP['left_hip'], 1],
        "hip_z": joints[JOINT_MAP['left_hip'], 2],
        "hip_visibility": 1.0,
    }
    pose_data.append(frame_data)

df = pd.DataFrame(pose_data)

# Save CSV
csv_path = f"/kaggle/working/results/{VIDEO_NAME}_pose_3d.csv"
df.to_csv(csv_path, index=False)
print(f"✅ Saved: {csv_path}")
print(f"\nDataFrame preview:")
print(df.head())
print(f"\nShape: {df.shape}")

✅ Loaded joints: (323, 45, 3)
✅ Saved: /kaggle/working/results/sample_swing2_pose_3d.csv

DataFrame preview:
   frame   wrist_x   wrist_y   wrist_z  wrist_visibility   elbow_x   elbow_y  \
0      0  0.067744 -0.154819  6.834346               1.0  0.019046 -0.400708   
1      1  0.070398 -0.157220  6.847096               1.0  0.020452 -0.403003   
2      2  0.071032 -0.156926  6.848421               1.0  0.019227 -0.402345   
3      3  0.073370 -0.157581  6.846765               1.0  0.019935 -0.402632   
4      4  0.075395 -0.157444  6.846387               1.0  0.021113 -0.402346   

    elbow_z  elbow_visibility  shoulder_x  shoulder_y  shoulder_z  \
0  6.862101               1.0    0.078735   -0.644737    6.780758   
1  6.873540               1.0    0.080049   -0.647089    6.792299   
2  6.874667               1.0    0.078567   -0.646613    6.793787   
3  6.873183               1.0    0.078445   -0.647127    6.792383   
4  6.872459               1.0    0.078187   -0.647547    6.792780

In [6]:
# ============================================
# CELL 5: RUN SWING ANALYSIS
# ============================================

import pandas as pd
import numpy as np
from scipy.signal import find_peaks, savgol_filter

VIDEO_NAME = "pro_swing6"

# Load pose data
df = pd.read_csv(f"/kaggle/working/results/{VIDEO_NAME}_pose_3d.csv")
print(f"✅ Loaded pose data: {df.shape}")


def analyze_swing(df):
    """
    Full swing analysis pipeline with robust phase detection.
    Returns analyzed DataFrame with swing metrics.
    """
    
    if df is None or df.empty:
        return pd.DataFrame()
    
    try:
        # -----------------------------------
        # 1. Smooth coordinates
        # -----------------------------------
        for joint in ["wrist", "elbow", "shoulder", "hip"]:
            for axis in ["x", "y", "z"]:
                col = f"{joint}_{axis}"
                if col in df.columns:
                    df[f"{col}_smooth"] = savgol_filter(
                        df[col].fillna(method='ffill').fillna(method='bfill'), 
                        window_length=7, 
                        polyorder=2, 
                        mode='nearest'
                    )

        y = df["wrist_y_smooth"].values

        # -----------------------------------
        # 2. Detect backswing start
        # -----------------------------------
        def detect_backswing_start_robust(y):
            y = np.asarray(y)

            total_range = np.percentile(y, 95) - np.percentile(y, 5)
            significant_drop = 0.30 * total_range

            window = 20
            rolling_std = (
                pd.Series(y)
                .rolling(window, center=True)
                .std()
                .bfill()
                .ffill()
            )

            max_search = min(len(y) // 2, 250)
            stability_threshold = 0.01
            min_stable_length = 12

            stable_segments = []
            current_start = None
            current_len = 0

            for i in range(window, max_search):
                if rolling_std.iloc[i] < stability_threshold:
                    if current_start is None:
                        current_start = i
                    current_len += 1
                else:
                    if current_start is not None and current_len >= min_stable_length:
                        stable_segments.append((current_start, i - 1, current_len))
                    current_start = None
                    current_len = 0

            if current_start is not None and current_len >= min_stable_length:
                stable_segments.append((current_start, max_search - 1, current_len))

            if stable_segments:
                stable_segments.sort(key=lambda x: x[2], reverse=True)
                stable_start, stable_end, _ = stable_segments[0]
                address_mean = np.mean(y[stable_start:stable_end])
            else:
                stable_end = 20
                address_mean = np.mean(y[:20])

            local_drop = 0.02
            confirm_window = 40

            for i in range(stable_end, min(stable_end + 60, len(y) - confirm_window)):
                if y[i] < address_mean - local_drop:
                    future_min = np.min(y[i:i + confirm_window])

                    if address_mean - future_min > significant_drop:
                        return max(i - 5, stable_end)

            dy = np.diff(y)
            for i in range(20, min(250, len(dy) - 6)):
                if np.all(dy[i:i + 6] < -0.01):
                    return i

            return 0

        backswing_start = detect_backswing_start_robust(y)

        # -----------------------------------
        # 3. Top of backswing 
        # -----------------------------------
        search_window = min(90, len(y) - backswing_start - 1)
        segment = y[backswing_start: backswing_start + search_window]

        troughs, properties = find_peaks(
            -segment,
            prominence=0.15 * (np.max(segment) - np.min(segment)),
            distance=8
        )

        if len(troughs) > 0:
            backswing_top = backswing_start + troughs[0]
        else:
            backswing_top = backswing_start + np.argmin(segment)

        print(f"DEBUG: Backswing start = {backswing_start}")
        print(f"DEBUG: Backswing top = {backswing_top}")
        print(f"DEBUG: Y at top = {y[backswing_top]:.4f}")

        # -----------------------------------
        # 4. Impact detection
        # Find where wrist Y returns to address level
        # -----------------------------------
        address_y = np.mean(y[max(0, backswing_start-5):backswing_start+5])
        print(f"DEBUG: Address Y level = {address_y:.4f}")
        
        search_start = backswing_top
        search_end = min(backswing_top + 40, len(y))
        
        downswing_y = y[search_start:search_end]
        impact_relative = np.argmin(np.abs(downswing_y - address_y))
        impact = search_start + impact_relative
        
        print(f"DEBUG: Impact = {impact}")
        print(f"DEBUG: Y at impact = {y[impact]:.4f}")

        # Save phase indices
        df.loc[:, "backswing_start_idx"] = backswing_start
        df.loc[:, "backswing_top_idx"] = backswing_top
        df.loc[:, "impact_idx"] = impact

        # -----------------------------------
        # 5. Hand Path Analysis (Steep vs Shallow) 
        # -----------------------------------
        wrist_y = df["wrist_y_smooth"].values
        wrist_z = df["wrist_z_smooth"].values
        
        # FIXED: Y increases as hands drop (SMPL Y-up coordinate system)
        # y_drop = how much Y increased from top to impact (positive = hands dropped correctly)
        y_drop = wrist_y[impact] - wrist_y[backswing_top]
        z_forward = wrist_z[impact] - wrist_z[backswing_top]

        if abs(z_forward) > 0.001:
            steepness_ratio = abs(y_drop / z_forward)  # Use abs to handle coordinate variations
        else:
            steepness_ratio = 1.0

        # Classify hand path
        if steepness_ratio > 1.5:
            hand_path_label = "Steep (good for irons)"
        elif steepness_ratio > 0.8:
            hand_path_label = "Neutral (versatile)"
        else:
            hand_path_label = "Shallow (good for driver)"

        df["hand_path_steepness"] = steepness_ratio
        df["hand_path_label"] = hand_path_label

        print(f"DEBUG: Y drop = {y_drop:.4f}, Z forward = {z_forward:.4f}")
        print(f"DEBUG: Hand path steepness = {steepness_ratio:.2f}")
        print(f"DEBUG: Hand path = {hand_path_label}")

        # -----------------------------------
        # 6. Arm Extension at Impact - FIXED
        # -----------------------------------
        def calculate_angle(p1, p2, p3):
            """Calculate angle at p2 formed by p1-p2-p3"""
            v1 = np.array(p1) - np.array(p2)
            v2 = np.array(p3) - np.array(p2)
            
            norm1 = np.linalg.norm(v1)
            norm2 = np.linalg.norm(v2)
            
            if norm1 < 1e-8 or norm2 < 1e-8:
                return 180.0  # Default to straight if vectors are too small
            
            cos_angle = np.dot(v1, v2) / (norm1 * norm2)
            cos_angle = np.clip(cos_angle, -1, 1)
            angle = np.degrees(np.arccos(cos_angle))
            return angle

        # Get joint positions at impact
        if all(col in df.columns for col in ["shoulder_x_smooth", "elbow_x_smooth", "wrist_x_smooth"]):
            shoulder_impact = [
                df.loc[impact, "shoulder_x_smooth"],
                df.loc[impact, "shoulder_y_smooth"],
                df.loc[impact, "shoulder_z_smooth"]
            ]
            elbow_impact = [
                df.loc[impact, "elbow_x_smooth"],
                df.loc[impact, "elbow_y_smooth"],
                df.loc[impact, "elbow_z_smooth"]
            ]
            wrist_impact = [
                df.loc[impact, "wrist_x_smooth"],
                df.loc[impact, "wrist_y_smooth"],
                df.loc[impact, "wrist_z_smooth"]
            ]
            
            # Elbow angle (180° = fully straight)
            elbow_angle = calculate_angle(shoulder_impact, elbow_impact, wrist_impact)
            
            # FIXED: Adjusted thresholds - SMPL angles may differ from real anatomy
            # Pro golfers typically have some bend, pure 180° is rare
            if elbow_angle > 155:
                arm_extension_label = "Excellent (fully extended)"
            elif elbow_angle > 140:
                arm_extension_label = "Good (slight bend is normal)"
            elif elbow_angle > 125:
                arm_extension_label = "Moderate"
            else:
                arm_extension_label = "Needs improvement (chicken wing)"
            
            df["elbow_angle_impact"] = elbow_angle
            df["arm_extension_label"] = arm_extension_label
            
            print(f"DEBUG: Elbow angle at impact = {elbow_angle:.1f}°")
            print(f"DEBUG: Arm extension = {arm_extension_label}")
        else:
            df["elbow_angle_impact"] = None
            df["arm_extension_label"] = "N/A (missing joint data)"

        # -----------------------------------
        # 7. Wrist Speed - FIXED
        # -----------------------------------
        wrist_x = df["wrist_x_smooth"].values
        
        # Calculate velocity
        dx = np.diff(wrist_x)
        dy = np.diff(wrist_y)
        dz = np.diff(wrist_z)
        velocity = np.sqrt(dx**2 + dy**2 + dz**2)
        
        # Find max velocity during downswing
        downswing_velocity = velocity[backswing_top:impact]
        if len(downswing_velocity) > 0:
            max_velocity = np.max(downswing_velocity)
            max_velocity_frame = backswing_top + np.argmax(downswing_velocity)
        else:
            max_velocity = 0
            max_velocity_frame = impact
        
        df["max_wrist_speed"] = max_velocity
        df["max_speed_frame"] = max_velocity_frame
        
        # FIXED: Speed efficiency calculation
        # For pros, max speed should be AT or JUST BEFORE impact (frames 60-63)
        # This is GOOD, not bad!
        
        # Check if max speed is near impact (within 5 frames)
        frames_from_impact = abs(max_velocity_frame - impact)
        
        if frames_from_impact <= 2:
            speed_timing = "Excellent (max speed at impact)"
            speed_timing_score = 100
        elif frames_from_impact <= 4:
            speed_timing = "Good (max speed near impact)"
            speed_timing_score = 85
        elif frames_from_impact <= 6:
            speed_timing = "Moderate (slightly early release)"
            speed_timing_score = 70
        else:
            speed_timing = "Early release (losing speed before impact)"
            speed_timing_score = 50
        
        df["speed_timing"] = speed_timing
        df["speed_timing_score"] = speed_timing_score
        df["max_speed_frames_from_impact"] = frames_from_impact
        
        print(f"DEBUG: Max wrist speed = {max_velocity:.4f} at frame {max_velocity_frame}")
        print(f"DEBUG: Frames from impact = {frames_from_impact}")
        print(f"DEBUG: Speed timing = {speed_timing}")

        # -----------------------------------
        # 8. Tempo calculation
        # -----------------------------------
        backswing_time = backswing_top - backswing_start
        downswing_time = impact - backswing_top
        tempo_ratio = round(backswing_time / downswing_time, 2) if downswing_time > 0 else 0.0

        df["tempo_ratio"] = tempo_ratio
        df["backswing_frames"] = backswing_time
        df["downswing_frames"] = downswing_time

        # -----------------------------------
        # 9. Finish detection
        # -----------------------------------
        if impact + 20 < len(y):
            post_impact_std = pd.Series(y[impact:]).rolling(10).std()
            stable_frames = np.where(post_impact_std < 0.02)[0]
            if len(stable_frames) > 0:
                finish = impact + stable_frames[0]
            else:
                finish = len(y) - 1
        else:
            finish = len(y) - 1
        
        df.loc[:, "finish_idx"] = finish
        df["follow_through_frames"] = finish - impact

        # -----------------------------------
        # 10. Overall Swing Rating - FIXED
        # -----------------------------------
        score = 0
        max_score = 0
        
        # Tempo (ideal: 2.0-3.5:1)
        max_score += 25
        if 2.0 <= tempo_ratio <= 3.5:
            score += 25
        elif 1.5 <= tempo_ratio <= 4.0:
            score += 18
        else:
            score += 10
        
        # Arm extension (adjusted thresholds)
        if "elbow_angle_impact" in df.columns and df["elbow_angle_impact"].iloc[0] is not None:
            max_score += 25
            elbow_angle = df["elbow_angle_impact"].iloc[0]
            if elbow_angle > 155:
                score += 25
            elif elbow_angle > 140:
                score += 22
            elif elbow_angle > 125:
                score += 15
            else:
                score += 8
        
        # Hand path (neutral is ideal)
        max_score += 25
        if 0.8 <= steepness_ratio <= 1.5:
            score += 25
        elif 0.5 <= steepness_ratio <= 2.0:
            score += 18
        else:
            score += 10
        
        # Speed timing (max speed should be near impact)
        max_score += 25
        score += int(speed_timing_score * 0.25)
        
        overall_rating = round(score / max_score * 100) if max_score > 0 else 0
        
        if overall_rating >= 85:
            rating_label = "Excellent"
        elif overall_rating >= 70:
            rating_label = "Good"
        elif overall_rating >= 55:
            rating_label = "Average"
        else:
            rating_label = "Needs Work"
        
        df["overall_score"] = overall_rating
        df["overall_rating"] = rating_label

        return df
        
    except Exception as e:
        print(f"Swing analysis failed: {e}")
        import traceback
        traceback.print_exc()
        return df


def print_analysis_results(df):
    """Print formatted analysis results"""
    print("\n" + "="*60)
    print("🏌️ SWING ANALYSIS RESULTS")
    print("="*60)
    
    print(f"\n📍 PHASE DETECTION:")
    print(f"   Address/Backswing Start: Frame {int(df['backswing_start_idx'].iloc[0])}")
    print(f"   Top of Backswing:        Frame {int(df['backswing_top_idx'].iloc[0])}")
    print(f"   Impact:                  Frame {int(df['impact_idx'].iloc[0])}")
    print(f"   Finish:                  Frame {int(df['finish_idx'].iloc[0])}")
    
    print(f"\n⏱️ TEMPO:")
    print(f"   Backswing:      {int(df['backswing_frames'].iloc[0])} frames")
    print(f"   Downswing:      {int(df['downswing_frames'].iloc[0])} frames")
    print(f"   Follow-through: {int(df['follow_through_frames'].iloc[0])} frames")
    print(f"   Tempo Ratio:    {df['tempo_ratio'].iloc[0]}:1")
    
    ratio = df['tempo_ratio'].iloc[0]
    if 2.0 <= ratio <= 3.5:
        print(f"   ✅ Good tempo (tour average is ~3:1)")
    elif ratio < 2.0:
        print(f"   ⚠️ Quick backswing - try slowing down")
    else:
        print(f"   ⚠️ Slow downswing - try accelerating through impact")
    
    print(f"\n🖐️ HAND PATH:")
    print(f"   Steepness Ratio: {df['hand_path_steepness'].iloc[0]:.2f}")
    print(f"   Classification:  {df['hand_path_label'].iloc[0]}")
    
    print(f"\n💪 ARM EXTENSION AT IMPACT:")
    if df['elbow_angle_impact'].iloc[0] is not None:
        print(f"   Elbow Angle:    {df['elbow_angle_impact'].iloc[0]:.1f}°")
        print(f"   Classification: {df['arm_extension_label'].iloc[0]}")
    else:
        print(f"   Data not available")
    
    print(f"\n⚡ SPEED TIMING:")
    print(f"   Max Speed Frame:    {int(df['max_speed_frame'].iloc[0])}")
    print(f"   Impact Frame:       {int(df['impact_idx'].iloc[0])}")
    print(f"   Frames Difference:  {int(df['max_speed_frames_from_impact'].iloc[0])}")
    print(f"   Assessment:         {df['speed_timing'].iloc[0]}")
    
    print(f"\n🏆 OVERALL RATING:")
    print(f"   Score:  {df['overall_score'].iloc[0]}/100")
    print(f"   Rating: {df['overall_rating'].iloc[0]}")
    
    print("="*60)


# Run analysis
df_analyzed = analyze_swing(df)

# Print results
print_analysis_results(df_analyzed)

# Save results
df_analyzed.to_csv(f"/kaggle/working/results/{VIDEO_NAME}_analysis.csv", index=False)
print(f"\n✅ Saved: /kaggle/working/results/{VIDEO_NAME}_analysis.csv")




FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/results/pro_swing6_pose_3d.csv'

In [ ]:
# ============================================
# CELL 6: VISUALIZE RESULTS
# ============================================

import matplotlib.pyplot as plt

VIDEO_NAME = "sample_swing2"
df = pd.read_csv(f"/kaggle/working/results/{VIDEO_NAME}_analysis.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Wrist Timeline
ax1 = axes[0]
ax1.plot(df["wrist_y_smooth"], label="Wrist Y", linewidth=2)
ax1.axvline(df["backswing_start_idx"].iloc[0], color="green", linestyle="--", label="Backswing Start")
ax1.axvline(df["backswing_top_idx"].iloc[0], color="orange", linestyle="--", label="Top")
ax1.axvline(df["impact_idx"].iloc[0], color="red", linestyle="--", label="Impact")
ax1.set_title("Wrist Height Timeline")
ax1.set_xlabel("Frame")
ax1.set_ylabel("Y Position")
ax1.legend()

# Plot 2: Bird's Eye View
ax2 = axes[1]
ax2.plot(df["wrist_z_smooth"], df["wrist_x_smooth"], linewidth=2)
ax2.set_title("Bird's Eye View (Swing Path)")
ax2.set_xlabel("Z (Forward/Back)")
ax2.set_ylabel("X (Side to Side)")
ax2.grid(True)

# Plot 3: Tempo
ax3 = axes[2]
bs = int(df["backswing_start_idx"].iloc[0])
top = int(df["backswing_top_idx"].iloc[0])
impact = int(df["impact_idx"].iloc[0])
ax3.bar(["Backswing", "Downswing"], [top - bs, impact - top], color=['blue', 'orange'])
ax3.set_title(f"Tempo Ratio: {df['tempo_ratio'].iloc[0]}:1")
ax3.set_ylabel("Frames")

plt.tight_layout()
plt.savefig(f"/kaggle/working/results/{VIDEO_NAME}_plots.png", dpi=150)
plt.show()

print(f"✅ Saved: /kaggle/working/results/{VIDEO_NAME}_plots.png")

In [ ]:
# ============================================
# CELL 7: CREATE SKELETON OVERLAY VIDEO
# ============================================

import cv2
from PIL import Image
import os

VIDEO_NAME = "pro_swing5"
VIDEO_PATH = f"/kaggle/working/{VIDEO_NAME}.mp4"

# Load joints
all_joints = np.load(f"/kaggle/working/results/{VIDEO_NAME}_joints.npy")

# Skeleton connections
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (0, 3), (1, 4), (2, 5), (4, 7), (5, 8),
    (7, 10), (8, 11), (3, 6), (6, 9), (9, 12), (9, 13), (9, 14),
    (12, 15), (13, 16), (14, 17), (16, 18), (17, 19),
    (18, 20), (19, 21), (20, 22), (21, 23)
]

# Open video
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
output_video = f'/kaggle/working/results/pro_swing2_skeleton.mp4'
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

gif_frames = []
frame_idx = 0
SCALE = 100  # Adjust if skeleton doesn't align

print(f"⏳ Processing {all_joints.shape[0]} frames...")

while frame_idx < all_joints.shape[0]:
    ret, frame = cap.read()
    if not ret:
        break
    
    joints_3d = all_joints[frame_idx]
    
    # Project to 2D
    joints_2d = joints_3d[:, [0, 1]].copy()
    joints_2d *= SCALE
    joints_2d[:, 0] += width / 2
    joints_2d[:, 1] += height / 2
    
    overlay = frame.copy()
    
    # Draw skeleton
    for (j1, j2) in SKELETON_CONNECTIONS:
        pt1 = (int(joints_2d[j1, 0]), int(joints_2d[j1, 1]))
        pt2 = (int(joints_2d[j2, 0]), int(joints_2d[j2, 1]))
        
        if (0 <= pt1[0] < width and 0 <= pt1[1] < height and
            0 <= pt2[0] < width and 0 <= pt2[1] < height):
            cv2.line(overlay, pt1, pt2, (0, 255, 0), 4)
    
    # Draw joints
    for (x, y) in joints_2d:
        if 0 <= x < width and 0 <= y < height:
            cv2.circle(overlay, (int(x), int(y)), 7, (0, 0, 255), -1)
    
    result = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)
    out.write(result)
    
    # GIF frames
    if frame_idx % 2 == 0:
        rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (width//2, height//2))
        gif_frames.append(Image.fromarray(rgb))
    
    frame_idx += 1
    if frame_idx % 20 == 0:
        print(f"  Frame {frame_idx}/{all_joints.shape[0]}")

cap.release()
out.release()

print(f"✅ Video saved: {output_video}")

# Create GIF
output_gif = f'/kaggle/working/results/{VIDEO_NAME}_skeleton.gif'
if gif_frames:
    gif_frames[0].save(output_gif, save_all=True, append_images=gif_frames[1:],
                       duration=int(1000/fps*2), loop=0)
    print(f"✅ GIF saved: {output_gif}")

# Display GIF
from IPython.display import Image as IPImage
IPImage(filename=output_gif)

In [ ]:
!cp /kaggle/input/smpl-model/SMPL_python_v.1.1.0/smpl/models/basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl \
/kaggle/working/ml-comotion-main/src/comotion_demo/data/smpl/SMPL_NEUTRAL.pkl

!pip install smplx

print("✅ smplx installed")

In [ ]:
%cd /kaggle/working/ml-comotion-main
!pip install -e .


In [ ]:
!cp /kaggle/input/project-with-model/FYP/Swing-motion-Analysis/Data/pro_swing2.mp4 /kaggle/working/


In [ ]:
!bash get_pretrained_models.sh



In [ ]:

!python /kaggle/working/ml-comotion-main/demo.py \
-i /kaggle/working/pro_swing2.mp4 \
-o /kaggle/working/results/ \
--skip-visualization



In [ ]:
import torch

# Path to the tracked pose results
pt_file = "/kaggle/working/results/pro_swing2.pt"

# Load the data
data = torch.load(pt_file, map_location="cpu")  # Always use map_location on Kaggle

# Inspect the keys
for k, v in data.items():
    print(k, type(v), v.shape if hasattr(v, "shape") else "")


In [ ]:
# print("Step 4: Preparing video...")

# !cp /kaggle/input/project-with-model/FYP/Swing-motion-Analysis/Data/pro_swing4.mp4 \
#    /kaggle/working/pro_swing4.mp4

# import cv2
# cap = cv2.VideoCapture('/kaggle/working/pro_swing4.mp4')
# fps = int(cap.get(cv2.CAP_PROP_FPS))
# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# cap.release()

# print(f"✅ Video copied")
# print(f"   Resolution: {width}x{height}")
# print(f"   FPS: {fps}")
# print(f"   Frames: {total_frames}")
# print(f"   Duration: {total_frames/fps:.2f}s")

In [ ]:
# import os

# print("\n" + "="*70)
# print("Step 5: Processing video with CoMotion...")
# print("="*70)

# os.chdir('/kaggle/working/ml-comotion-main')

# # Run CoMotion demo
# !python demo.py \
#   -i /kaggle/working/pro_swing4.mp4 \
#   -o /kaggle/working/results/ \
#   --skip-visualization

# print("\n✅ Processing complete!")


In [ ]:
import torch
import pickle

data = torch.load(
    "/kaggle/working/results/pro_swing2.pt",
    map_location="cpu",
    pickle_module=pickle
)

print(data.keys())


In [ ]:
print("Step 7: Extracting 3D joint positions...")

import torch
import pandas as pd
import numpy as np
from smplx import SMPL

# Load CoMotion output
data = torch.load('/kaggle/working/results/pro_swing2.pt', map_location='cpu')

print(f"✅ Loaded .pt file")
print(f"   Keys: {list(data.keys())}")
print(f"   Frames: {data['pose'].shape[0]}")

# Extract data
pose_tensor = data['pose']
trans_tensor = data['trans']
betas_tensor = data['betas']

global_orient = pose_tensor[:, :3]
body_pose = pose_tensor[:, 3:72]

# Initialize SMPL
smpl_model = SMPL(
    model_path='/kaggle/working/ml-comotion-main/src/comotion_demo/data/smpl',
    gender='neutral',
    create_transl=True
).to('cpu')

print("✅ SMPL model initialized")

# Extract joints
print("\nExtracting joints...")
all_joints = []

for frame_idx in range(pose_tensor.shape[0]):
    if frame_idx % 10 == 0:
        print(f"  Frame {frame_idx}/{pose_tensor.shape[0]}")
    
    smpl_out = smpl_model(
        body_pose=body_pose[frame_idx].unsqueeze(0),
        global_orient=global_orient[frame_idx].unsqueeze(0),
        betas=betas_tensor[frame_idx].unsqueeze(0),
        transl=trans_tensor[frame_idx].unsqueeze(0)
    )
    
    joints = smpl_out.joints[0].detach().cpu().numpy()
    all_joints.append(joints)

all_joints = np.array(all_joints)

print(f"\n✅ Joints extracted: {all_joints.shape}")

In [ ]:
print("Creating skeleton overlay video...")
import os
import cv2
from PIL import Image

# Skeleton connections
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (0, 3), (1, 4), (2, 5), (4, 7), (5, 8),
    (7, 10), (8, 11), (3, 6), (6, 9), (9, 12), (9, 13), (9, 14),
    (12, 15), (13, 16), (14, 17), (16, 18), (17, 19),
    (18, 20), (19, 21), (20, 22), (21, 23)
]

# Open video
cap = cv2.VideoCapture('/kaggle/working/pro_swing2.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

output_video = '/kaggle/working/results/skeleton_overlay.mp4'
output_gif = '/kaggle/working/results/skeleton_overlay.gif'

out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

gif_frames = []
frame_idx = 0

SCALE = 100  # pixels per meter (adjust if skeleton doesn't align)

print(f"Processing {all_joints.shape[0]} frames...")

while frame_idx < all_joints.shape[0]:
    ret, frame = cap.read()
    if not ret:
        break
    
    joints_3d = all_joints[frame_idx]
    
    # Project to 2D (no flip Y - correct orientation)
    joints_2d = joints_3d[:, [0, 1]].copy()
    joints_2d *= SCALE
    joints_2d[:, 0] += width / 2
    joints_2d[:, 1] += height / 2
    
    overlay = frame.copy()
    
    # Draw skeleton connections
    for (j1, j2) in SKELETON_CONNECTIONS:
        pt1 = (int(joints_2d[j1, 0]), int(joints_2d[j1, 1]))
        pt2 = (int(joints_2d[j2, 0]), int(joints_2d[j2, 1]))
        
        if (0 <= pt1[0] < width and 0 <= pt1[1] < height and
            0 <= pt2[0] < width and 0 <= pt2[1] < height):
            cv2.line(overlay, pt1, pt2, (0, 255, 0), 4)
    
    # Draw joints
    for (x, y) in joints_2d:
        if 0 <= x < width and 0 <= y < height:
            cv2.circle(overlay, (int(x), int(y)), 7, (0, 0, 255), -1)
    
    # Blend
    result = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)
    
    out.write(result)
    
    # GIF frames (every 2nd frame)
    if frame_idx % 2 == 0:
        rgb_frame = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
        rgb_frame = cv2.resize(rgb_frame, (width//2, height//2))
        gif_frames.append(Image.fromarray(rgb_frame))
    
    frame_idx += 1
    if frame_idx % 10 == 0:
        print(f"  Frame {frame_idx}/{all_joints.shape[0]}")

cap.release()
out.release()

print(f"\n✅ Video saved: {output_video}")

# Create GIF
if gif_frames:
    gif_frames[0].save(
        output_gif,
        save_all=True,
        append_images=gif_frames[1:],
        duration=int(1000/fps*2),
        loop=0
    )
    size_mb = os.path.getsize(output_gif) / (1024*1024)
    print(f"✅ GIF saved: {output_gif} ({size_mb:.2f} MB)")

print("\n✅ Done! Check /kaggle/working/results/")

In [ ]:
from IPython.display import Image as IPImage

IPImage(filename='/kaggle/working/results/skeleton_overlay.gif')

In [ ]:
# Create CSV
print("Creating CSV...")

joint_names = [
    'pelvis', 'left_hip', 'right_hip', 'spine1', 'left_knee', 'right_knee',
    'spine2', 'left_ankle', 'right_ankle', 'spine3', 'left_foot', 'right_foot',
    'neck', 'left_collar', 'right_collar', 'head', 'left_shoulder', 'right_shoulder',
    'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hand', 'right_hand'
]

data_list = []
for frame_idx in range(all_joints.shape[0]):
    frame_data = {'frame': frame_idx}
    for j_idx, j_name in enumerate(joint_names):
        if j_idx < all_joints.shape[1]:
            frame_data[f'{j_name}_x'] = all_joints[frame_idx, j_idx, 0]
            frame_data[f'{j_name}_y'] = all_joints[frame_idx, j_idx, 1]
            frame_data[f'{j_name}_z'] = all_joints[frame_idx, j_idx, 2]
    data_list.append(frame_data)

df = pd.DataFrame(data_list)
df.to_csv('/kaggle/working/results/golf_swing2_3d_comotion.csv', index=False)

print(f"✅ CSV saved")
print(f"   Shape: {df.shape}")
print(df.head())

In [ ]:
from IPython.display import Image as IPImage

# Display the GIF
IPImage(filename="/kaggle/working/results/skeleton_final.gif")

In [ ]:
!streamlit run app.py


